# 19 — Evaluasi ulang saja (opsional)

**Alur utama sudah digabung di notebook 18. Lewati notebook ini jika evaluasi 18 selesai.**
Gunakan hanya jika ingin melanjutkan RAGAS atau menilai kembali prediksi tersimpan, tanpa
mengunduh empat GGUF, tanpa retrieval/generasi ulang, dan tanpa menyalakan aplikasi.
Attach folder output notebook 18 sebagai Kaggle input (ekstrak ZIP terlebih dahulu jika perlu).
Isi INPUT_RUN_DIR tepat ke folder berisi manifest.json. Output disalin ke working, sehingga
input read-only dan hasil lama tidak ditimpa. Metrik sama persis dengan notebook 18.

In [ ]:
import subprocess, sys
# Jalankan pada sesi Kaggle baru sebelum mengimpor pustaka lain.
INSTALL_DEPENDENCIES = True
if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
        "numpy==1.26.4", "pandas==2.2.3", "requests>=2.32,<3",
        "huggingface-hub==0.27.1",
        "langchain==0.3.14", "langchain-core==0.3.29", "datasets==3.2.0",
        "openai==1.59.7", "tiktoken==0.8.0",
        "langchain-community==0.3.14", "langchain-openai==0.2.14",
        "ragas==0.2.15"])
# PyTorch/CUDA bawaan Kaggle tidak dipasang ulang. Jika diminta, restart kernel
# setelah instalasi lalu ubah INSTALL_DEPENDENCIES=False dan Run All kembali.

In [ ]:
from pathlib import Path
import os, json, hashlib, sys, platform, time, random
import importlib.metadata as metadata
import pandas as pd
import numpy as np

# Hitung identitas isi artefak untuk mencegah pencampuran eksperimen.
def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()

# Simpan JSON secara atomik agar berkas tidak setengah tertulis.
def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    temp.replace(path)


# Ambil secret tanpa mencetak nilainya atau menyimpannya dalam artefak.
def read_secret(name, required=False):
    value = os.environ.get(name)
    if not value:
        try:
            from kaggle_secrets import UserSecretsClient
            value = UserSecretsClient().get_secret(name)
        except Exception:
            value = None
    if required and not value:
        raise RuntimeError(f"Aktifkan secret {name} pada Kaggle Add-ons > Secrets.")
    return value

HF_TOKEN = read_secret("HF_Faiss")
# OpenAI hanya dipanggil ketika RUN_RAGAS=True; key tidak ditulis ke notebook.

In [ ]:
import shutil
INPUT_RUN_DIR = Path("/kaggle/input/GANTI-DENGAN-OUTPUT-18/pilot_v2")
RUN_NAME = "reeval_v1"
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
OUT = WORK / "rag_results" / RUN_NAME
SEED = 42
RUN_RAGAS = False  # True mengizinkan API OpenAI berbayar; aktifkan secret OPENAI_API_KEY.
JUDGE_MODEL = "gpt-4o-mini-2024-07-18"
JUDGE_EMBEDDING = "text-embedding-3-small"
assert (INPUT_RUN_DIR / "manifest.json").is_file(), "Isi INPUT_RUN_DIR dengan folder hasil 18."
assert OUT.resolve() != INPUT_RUN_DIR.resolve()
OUT.mkdir(parents=True, exist_ok=True)
if (OUT / "manifest.json").exists():
    assert (OUT / "manifest.json").read_bytes() == (INPUT_RUN_DIR / "manifest.json").read_bytes(), "Run berbeda."
for name in ["manifest.json", "test_selected.csv", "predictions", "ragas_cache", "ragas_evaluators"]:
    source, target = INPUT_RUN_DIR / name, OUT / name
    if not source.exists():
        continue
    if source.is_dir():
        for path in source.rglob("*"):
            if path.is_file():
                dest = target / path.relative_to(source)
                dest.parent.mkdir(parents=True, exist_ok=True)
                if not dest.exists():
                    shutil.copy2(path, dest)
    elif not target.exists():
        shutil.copy2(source, target)


In [ ]:
# Baca prediksi dan pastikan hasil berasal dari satu eksperimen yang sama.
def load_predictions():
    manifest = json.loads((OUT / "manifest.json").read_text(encoding="utf-8"))
    rows = [json.loads(p.read_text(encoding="utf-8")) for p in sorted((OUT / "predictions").glob("*.json"))]
    if not rows:
        raise ValueError("Jalankan notebook 18 terlebih dahulu.")
    assert all(r["experiment_id"] == manifest["experiment_id"] for r in rows)
    df = pd.DataFrame(rows)
    assert not df.duplicated(["record_id", "model_key", "condition", "protocol"]).any()
    return df, manifest

predictions, manifest = load_predictions()
display(predictions.groupby(["protocol", "model_key", "condition", "status"]).size().rename("n").reset_index())
# Siapkan masukan RAGAS tanpa jawaban acuan, qrels, atau metrik tahap sebelumnya.
def prepare_ragas_inputs(df):
    required = {"record_id", "model_key", "architecture", "condition", "protocol", "status"}
    if not required <= set(df):
        raise ValueError("Identitas/status prediksi tidak lengkap.")
    scored = df[df.status.eq("ok")].copy().reset_index(drop=True)
    if scored.empty:
        raise ValueError("Tidak ada prediksi berhasil; periksa inference_coverage.csv.")
    if not {"question", "answer", "contexts", "use_rag", "context_hash"} <= set(scored):
        raise ValueError("Masukan RAGAS tidak lengkap.")
    assert scored.question.map(lambda x: isinstance(x, str) and bool(x.strip())).all()
    assert scored.answer.map(lambda x: isinstance(x, str)).all()
    assert scored.contexts.map(lambda x: isinstance(x, list) and all(isinstance(y, str) for y in x)).all()
    assert scored.use_rag.map(lambda x: isinstance(x, (bool, np.bool_))).all()
    assert scored.loc[~scored.use_rag, "contexts"].map(len).eq(0).all(), "Tanpa RAG harus tanpa konteks."
    # Allowlist memastikan skor/kolom acuan lama tidak masuk ekspor baru.
    columns = ["record_id", "model_key", "architecture", "condition", "protocol", "status",
               "question", "answer", "contexts", "use_rag", "context_hash", "experiment_id", "subtask_type"]
    scored = scored[[c for c in columns if c in scored]].copy()
    if "subtask_type" not in scored:
        scored["subtask_type"] = ""
    return scored

predictions.groupby(["protocol", "architecture", "condition", "status"]).size().rename("n").to_csv(
    OUT / "inference_coverage.csv")
scored = prepare_ragas_inputs(predictions)


## RAGAS — tiga penilaian, OpenAI sebagai judge

1. **Faithfulness**: apakah klaim jawaban didukung konteks retrieval yang masuk prompt?
2. **Answer relevance** (`answer_relevancy` / `ResponseRelevancy`): kesesuaian jawaban dengan pertanyaan,
   memakai pertanyaan hasil rekonstruksi dan embedding OpenAI; strictness=3.
3. **Context relevance** (`context_relevance` / `ContextRelevance`): kesesuaian konteks dengan pertanyaan.

RAGAS 0.2.15 menyediakan ContextRelevance pada keluarga metrik NVIDIA: dua penilaian 0/1/2,
dinormalisasi menjadi 0–1 dan dirata-ratakan. **Ini bukan rumus fraksi kalimat relevan pada paper
RAGAS awal, bukan context precision, dan bukan context recall.** Definisi operasional ini harus
ditulis konsisten di Bab III. Tidak membutuhkan qrels atau kolom recall pada dataset.
Audit kode wheel 0.2.15 menemukan batas internal 7.000 karakter konteks, temperature=0.1,
dan template pertama dipanggil dua kali (berbeda dari uraian dokumentasi). Implementasi paket
dipertahankan; context_relevance_truncated mencatat pemotongan. Jangan mengklaim seluruh
konteks panjang dinilai. Audit dampak batas ini sebelum final.

Sumber: [faithfulness](https://docs.ragas.io/en/v0.2.15/concepts/metrics/available_metrics/faithfulness/),
[answer relevance](https://docs.ragas.io/en/v0.2.15/concepts/metrics/available_metrics/answer_relevance/),
[context relevance](https://docs.ragas.io/en/v0.2.15/concepts/metrics/available_metrics/nvidia_metrics/).

Aktifkan secret `OPENAI_API_KEY`, lalu `RUN_RAGAS=True`. Pertanyaan, jawaban dan potongan dokumen
dikirim ke OpenAI untuk penilaian; model yang diuji tetap GGUF lokal Kaggle, bukan OpenAI.
Ada biaya chat + embedding, dan satu metrik dapat memanggil API beberapa kali. Mulai dari pilot.
RAGAS tidak melatih ulang model. Nilai error/NaN dan empty retrieval dilaporkan bersama cakupan,
bukan diubah menjadi nol atau dianggap berhasil. Faithfulness/context relevance tidak berlaku pada
K1/K3; answer relevance tetap dihitung. Audit kecakapan judge pada bahasa Indonesia secara manual.

In [ ]:
# Susun evaluator OpenAI eksplisit; key hanya berada di memori proses.
def make_ragas_evaluators():
    assert metadata.version("ragas") == "0.2.15", "Gunakan versi paket yang dibekukan."
    from ragas.metrics import Faithfulness, ResponseRelevancy, ContextRelevance
    from ragas.llms import LangchainLLMWrapper
    from ragas.embeddings import LangchainEmbeddingsWrapper
    from ragas.run_config import RunConfig
    from langchain_openai import ChatOpenAI, OpenAIEmbeddings
    key = read_secret("OPENAI_API_KEY", required=True)
    judge = LangchainLLMWrapper(ChatOpenAI(model=JUDGE_MODEL, api_key=key, temperature=0,
        max_tokens=4096, timeout=120, max_retries=2), run_config=RunConfig(timeout=180, max_retries=2))
    embedding = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model=JUDGE_EMBEDDING, api_key=key,
        request_timeout=120, max_retries=2))
    return {"faithfulness": Faithfulness(llm=judge),
            "answer_relevancy": ResponseRelevancy(llm=judge, embeddings=embedding, strictness=3),
            "context_relevance": ContextRelevance(llm=judge)}

# Nilai tiap metrik dengan cache terpisah; kegagalan tidak mengulang metrik yang berhasil.
async def score_ragas_cached(df, metrics, config, sample_factory):
    import asyncio
    evaluator_id = digest(config)
    write_json(OUT / "ragas_evaluators" / (evaluator_id + ".json"), config)
    results = []
    for row in df.to_dict("records"):
        context_chars = len('\n'.join(row['contexts']))
        result = {"ragas_evaluator_id": evaluator_id,
                  "context_relevance_context_chars": context_chars,
                  "context_relevance_truncated": bool(row['use_rag'] and context_chars > 7000)}
        for name, metric in metrics.items():
            if name != "answer_relevancy" and (not row["use_rag"] or not row["contexts"]):
                result[name] = np.nan
                result[name + "_status"] = "no_rag_not_applicable" if not row["use_rag"] else "empty_retrieval"
                continue
            inputs = {"user_input": row["question"]}
            if name != "context_relevance":
                inputs["response"] = row["answer"]
            if name != "answer_relevancy":
                inputs["retrieved_contexts"] = row["contexts"]
            cache_key = digest({"evaluator": evaluator_id, "metric": name, "inputs": inputs})
            path = OUT / "ragas_cache" / (cache_key + ".json")
            cached = json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}
            if cached.get("status") != "ok":
                try:
                    value = float(await asyncio.wait_for(metric.single_turn_ascore(sample_factory(**inputs)), timeout=600))
                    cached = {"value": value if np.isfinite(value) else None,
                              "status": "ok" if np.isfinite(value) else "judge_nan"}
                except Exception as exc:
                    # Jangan simpan raw exception provider yang dapat memuat rincian kredensial.
                    cached = {"value": None, "status": "error_" + type(exc).__name__}
                write_json(path, cached)
            result[name] = np.nan if cached["value"] is None else cached["value"]
            result[name + "_status"] = cached["status"]
        results.append(result)
    return pd.DataFrame(results)

# Jalankan tiga metrik RAGAS terhadap prompt context aktual, bukan konteks jawaban acuan.
async def evaluate_ragas_rows(df):
    from ragas.dataset_schema import SingleTurnSample
    metrics = make_ragas_evaluators()
    config = {"implementation": "openai_three_metrics_v1", "judge": JUDGE_MODEL,
        "embedding": JUDGE_EMBEDDING, "configured_chat_temperature": 0, "max_tokens": 4096,
        "context_relevance_internal_temperature": 0.1, "context_relevance_max_chars": 7000,
        "context_relevance_prompts": "0.2.15 wheel repeats template_relevance1 twice",
        "strictness": 3, "prompt_language": "English default; audit Indonesian quality",
        "context_relevance_definition": "ragas NVIDIA ContextRelevance: two 0/1/2 judgments normalized",
        "versions": {n: metadata.version(n) for n in ["ragas", "langchain-openai", "openai"]}}
    return await score_ragas_cached(df, metrics, config, SingleTurnSample)

if RUN_RAGAS:
    ragas_scores = await evaluate_ragas_rows(scored)
    scored = pd.concat([scored.reset_index(drop=True), ragas_scores], axis=1)
else:
    for metric in ["faithfulness", "answer_relevancy", "context_relevance"]:
        scored[metric] = np.nan
        scored[metric + "_status"] = "not_run"
    print("RAGAS belum dijalankan. Aktifkan RUN_RAGAS=True setelah secret dan biaya disetujui.")

## Ringkasan RAGAS, kelengkapan, dan selisih berpasangan

Rata-rata tidak cukup untuk menyimpulkan pengaruh komponen: laporkan selisih pada record_id yang sama,
jumlah pasangan lengkap, kegagalan, serta distribusi menurut jenis tugas. Perbandingan end-to-end K4–K2
mengandung perubahan ekspansi kueri; perbandingan controlled K4–K2 menggunakan konteks identik.
Ringkasan ini deskriptif, tidak menganggap pertanyaan dari bagian dokumen yang sama saling independen.

In [ ]:
METRICS = ["faithfulness", "answer_relevancy", "context_relevance"]
GROUP = ["protocol", "architecture", "condition"]

# Ringkas setiap metrik beserta penyebut valid, simpangan baku, dan median.
def summarize(df, groups):
    return df.groupby(groups, dropna=False)[METRICS].agg(["mean", "count", "std", "median"])

# Hitung selisih kondisi hanya pada pertanyaan yang sama-sama berhasil dinilai.
def paired_differences(df):
    rows = []
    for (protocol, arch), group in df.groupby(["protocol", "architecture"]):
        contrasts = [("K2", "K1"), ("K3", "K1"), ("K4", "K2"), ("K4", "K3")]
        if protocol == "controlled":
            contrasts = [("K4", "K2")]
        for metric in METRICS:
            table = group.pivot(index="record_id", columns="condition", values=metric)
            for after, before in contrasts:
                if not {after, before} <= set(table):
                    continue
                if metric != "answer_relevancy" and ({after, before} & {"K1", "K3"}):
                    continue  # Jangan membandingkan metrik konteks dengan kondisi tanpa RAG.
                pair = table[[after, before]].dropna()
                if pair.empty:
                    continue
                if protocol == "controlled":
                    hashes = group.pivot(index="record_id", columns="condition", values="context_hash").loc[pair.index]
                    assert hashes[after].eq(hashes[before]).all(), "Konteks controlled berbeda!"
                delta = pair[after] - pair[before]
                rows.append({"protocol": protocol, "architecture": arch, "contrast": after+"-"+before,
                             "metric": metric, "n_pairs": len(pair), "mean_delta": delta.mean(),
                             "median_delta": delta.median()})
    return pd.DataFrame(rows, columns=["protocol", "architecture", "contrast", "metric", "n_pairs", "mean_delta", "median_delta"])

summary = summarize(scored, GROUP)
summary.to_csv(OUT / "system_metrics_summary.csv")
summarize(scored, GROUP + ["subtask_type"]).to_csv(OUT / "system_metrics_by_task.csv")
paired = paired_differences(scored)
paired.to_csv(OUT / "paired_differences.csv", index=False)
# JSON preserves list contexts; CSV detail intentionally contains scalar fields only.
scored.to_json(OUT / "system_scored.jsonl", orient="records", lines=True, force_ascii=False)
scalar_columns = [c for c in scored if not scored[c].map(lambda v: isinstance(v, (dict, list))).any()]
scored[scalar_columns].to_csv(OUT / "system_scored.csv", index=False)
coverage = predictions.groupby(GROUP + ["status"]).size().rename("n").reset_index()
coverage.to_csv(OUT / "inference_coverage.csv", index=False)
selected = pd.read_csv(OUT / "test_selected.csv", dtype=str)
expected = len(selected) * sum(
    1 for p in manifest["protocols"] for scenario in manifest["scenarios"]
    if p != "controlled" or scenario["use_rag"])
print("Prediksi tersimpan / diharapkan:", len(predictions), "/", expected)
print("PILOT — bukan hasil final skripsi" if manifest["pilot"] else "Full test configuration")
display(summary)
display(paired)

for metric in ["faithfulness", "answer_relevancy", "context_relevance"]:
    scored.groupby(GROUP + [metric + "_status"]).size().rename("n").to_csv(OUT / f"{metric}_coverage.csv")
write_json(OUT / "evaluation_scope.json", {"scope": "ragas_only_v1", "metrics": METRICS,
    "reference_answer_required": False, "run_ragas": RUN_RAGAS})


In [ ]:
import shutil
archive_path = shutil.make_archive(str(OUT.parent / RUN_NAME), "zip", root_dir=OUT.parent, base_dir=RUN_NAME)
print("Unduh seluruh folder/ZIP melalui panel Output Kaggle:", archive_path)
print("Tidak ada unggahan Hugging Face otomatis. Simpan output sebelum sesi Kaggle berakhir.")